# SE446 Big Data Engineering - Milestone 2 Spark ML

## Chicago Crime Analytics and Arrest Prediction

**Course:** SE446 - Big Data Engineering  
**Project:** Chicago Crime Analytics  
**Group:** Group 44  
**Student:** Abdulrahman Meir  

---

## Notebook Purpose

This notebook documents Milestone 2 using Apache Spark and Spark MLlib. It covers:

1. Spark analytics for the Chicago Crimes dataset.
2. Feature engineering for arrest prediction.
3. Training and evaluating three ML models.
4. Model comparison.
5. Feature importance and interpretation.
6. Evidence for local and YARN cluster execution.

The final corrected ML feature vector is:

```text
features[0] = District
features[1] = crime_index created from Primary Type
features[2] = Hour extracted from Date
features[3] = domestic_index created from Domestic
```

# 1. Spark Session Setup

This section initializes Spark. On the cluster, Spark should run using YARN.  
For local testing, the master can be `local[*]`.

The project execution evidence includes both:

- `Spark Master: yarn`
- `Spark Master: local[*]`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, to_timestamp, desc
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
import time

spark = (
    SparkSession.builder
    .appName("SE446_Milestone2_Group44_Notebook")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark Version:", spark.version)
print("Spark Master:", spark.sparkContext.master)
print("Application ID:", spark.sparkContext.applicationId)

# 2. Dataset Loading

The dataset is stored in HDFS.

For ML training, the sample dataset is used because training three models on the full dataset may exceed the cluster memory limit.

```text
hdfs:///data/chicago_crimes_sample.csv
```

A separate full dataset YARN check is also included in the repository:

```text
full_dataset_phaseA_check.py
output/milestone2/full_dataset_phaseA_yarn_output.txt
```

In [ ]:
input_path = "hdfs:///data/chicago_crimes_sample.csv"

df = spark.read.csv(input_path, header=True, inferSchema=True)

print("Input path:", input_path)
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

df.printSchema()
df.show(5, truncate=False)

# 3. Task 1 - Crime Type Distribution

This task counts how many crimes occurred for each `Primary Type`.

In [ ]:
crime_type_counts = (
    df.groupBy("Primary Type")
    .count()
    .orderBy(desc("count"))
)

crime_type_counts.show(10, truncate=False)

# 4. Task 2 - Location Hotspots

This task finds the most common crime locations using `Location Description`.

In [ ]:
location_hotspots = (
    df.groupBy("Location Description")
    .count()
    .orderBy(desc("count"))
)

location_hotspots.show(10, truncate=False)

# 5. Task 3 - Crime Trend Over Years

This task counts crimes by year. The dataset already includes a `Year` column, so it is used for analytics only.

Important note: `Year` is **not** used in the final ML feature vector. The ML pipeline uses `Hour` extracted from `Date`.

In [ ]:
yearly_trends = (
    df.groupBy("Year")
    .count()
    .orderBy("Year")
)

yearly_trends.show(truncate=False)

# 6. Task 4 - Arrest Rate Analysis

This task counts arrest and non-arrest records.

In [ ]:
arrest_distribution = (
    df.groupBy("Arrest")
    .count()
    .orderBy("Arrest")
)

arrest_distribution.show(truncate=False)

# 7. Task 5 - Feature Engineering Pipeline

The final corrected feature vector follows the milestone requirement:

```text
features[0] = District
features[1] = crime_index created from Primary Type
features[2] = Hour extracted from Date
features[3] = domestic_index created from Domestic
```

The target label is:

```text
label = Arrest
```

where:

```text
1 = arrest
0 = no arrest
```

In [ ]:
df_ml = df.select(
    col("Primary Type"),
    col("District"),
    col("Date"),
    col("Domestic"),
    col("Arrest")
)

df_ml = df_ml.filter(col("Arrest").isNotNull())

# Extract Hour from Date.
# Chicago crime dates are usually formatted like: 01/01/2024 12:30:00 AM
df_ml = df_ml.withColumn(
    "ParsedDate",
    to_timestamp(col("Date"), "MM/dd/yyyy hh:mm:ss a")
)

df_ml = df_ml.withColumn("Hour", hour(col("ParsedDate")))

# Clean missing values so the ML pipeline can run safely.
df_ml = df_ml.fillna({
    "Primary Type": "UNKNOWN",
    "District": -1,
    "Hour": -1,
    "Domestic": False
})

# Create binary label and numeric domestic feature.
df_ml = df_ml.withColumn("label", col("Arrest").cast("integer"))
df_ml = df_ml.withColumn("domestic_index", col("Domestic").cast("integer"))

# Keep valid binary labels only.
df_ml = df_ml.filter((col("label") == 0) | (col("label") == 1))

print("Rows after cleaning:", df_ml.count())

print("Label distribution:")
df_ml.groupBy("label").count().orderBy("label").show()

crime_indexer = StringIndexer(
    inputCol="Primary Type",
    outputCol="crime_index",
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=["District", "crime_index", "Hour", "domestic_index"],
    outputCol="features",
    handleInvalid="keep"
)

feature_pipeline = Pipeline(stages=[crime_indexer, assembler])

feature_model = feature_pipeline.fit(df_ml)
featured_df = feature_model.transform(df_ml)

featured_df.select(
    "Primary Type",
    "District",
    "Hour",
    "Domestic",
    "crime_index",
    "domestic_index",
    "features",
    "label"
).show(5, truncate=False)

# 8. Train/Test Split

The data is split into 80% training and 20% testing.

In [ ]:
train_data, test_data = featured_df.randomSplit([0.8, 0.2], seed=42)

train_data = train_data.cache()
test_data = test_data.cache()

print("Training rows:", train_data.count())
print("Testing rows:", test_data.count())

# 9. Task 6 - Train and Evaluate Three Models

The following models are trained:

1. Logistic Regression
2. Random Forest
3. Gradient-Boosted Trees

The parameters are kept stable for the university YARN environment to avoid memory termination while still demonstrating the required Spark ML workflow.

In [ ]:
def evaluate_model(model_name, predictions, training_time):
    auc_evaluator = BinaryClassificationEvaluator(
        labelCol="label",
        rawPredictionCol="rawPrediction",
        metricName="areaUnderROC"
    )

    accuracy_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )

    f1_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1"
    )

    precision_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedPrecision"
    )

    recall_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedRecall"
    )

    auc = auc_evaluator.evaluate(predictions)
    accuracy = accuracy_evaluator.evaluate(predictions)
    f1 = f1_evaluator.evaluate(predictions)
    precision = precision_evaluator.evaluate(predictions)
    recall = recall_evaluator.evaluate(predictions)

    print("=" * 80)
    print(f"MODEL RESULTS: {model_name}")
    print("=" * 80)
    print(f"AUC-ROC:       {auc:.4f}")
    print(f"Accuracy:      {accuracy:.4f}")
    print(f"F1 Score:      {f1:.4f}")
    print(f"Precision:     {precision:.4f}")
    print(f"Recall:        {recall:.4f}")
    print(f"Training Time: {training_time:.2f} seconds")

    print("\nConfusion Matrix:")
    predictions.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

    return {
        "model": model_name,
        "auc": auc,
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "training_time": training_time
    }


models = [
    (
        "Logistic Regression",
        LogisticRegression(
            featuresCol="features",
            labelCol="label",
            maxIter=50,
            regParam=0.01
        )
    ),
    (
        "Random Forest",
        RandomForestClassifier(
            featuresCol="features",
            labelCol="label",
            numTrees=30,
            maxDepth=5,
            seed=42
        )
    ),
    (
        "Gradient-Boosted Trees",
        GBTClassifier(
            featuresCol="features",
            labelCol="label",
            maxIter=20,
            maxDepth=5,
            seed=42
        )
    )
]

results = []
trained_models = {}

for model_name, estimator in models:
    print("\nTraining:", model_name)
    start_time = time.time()

    model = estimator.fit(train_data)

    training_time = time.time() - start_time
    predictions = model.transform(test_data)

    metrics = evaluate_model(model_name, predictions, training_time)

    results.append(metrics)
    trained_models[model_name] = model

# 10. Three-Model Comparison Table

The final successful run in `output/milestone2/m2_spark_ml_client_output.txt` produced the following results:

| Model | AUC | Accuracy | F1 Score | Precision | Recall | Training Time |
|---|---:|---:|---:|---:|---:|---:|
| Logistic Regression | 0.6530 | 0.8626 | 0.8061 | 0.8029 | 0.8626 | 16.12s |
| Random Forest | 0.7370 | 0.8844 | 0.8529 | 0.8755 | 0.8844 | 16.75s |
| Gradient-Boosted Trees | 0.7423 | 0.8803 | 0.8518 | 0.8618 | 0.8803 | 133.66s |

Based on AUC-ROC, the best model was:

```text
Gradient-Boosted Trees with AUC = 0.7423
```

In [ ]:
print(f"{'Model':<25} {'AUC':<10} {'Accuracy':<10} {'F1':<10} {'Precision':<10} {'Recall':<10} {'Time(s)':<10}")

for result in results:
    print(
        f"{result['model']:<25} "
        f"{result['auc']:<10.4f} "
        f"{result['accuracy']:<10.4f} "
        f"{result['f1']:<10.4f} "
        f"{result['precision']:<10.4f} "
        f"{result['recall']:<10.4f} "
        f"{result['training_time']:<10.2f}"
    )

best_model = max(results, key=lambda item: item["auc"])
print("\nBest model based on AUC-ROC:")
print(best_model["model"], f"with AUC = {best_model['auc']:.4f}")

# 11. Task 7 - Feature Importance and Interpretation

Random Forest feature importance is used to interpret which feature contributed most to prediction.

From the final successful YARN run:

| Feature | Importance |
|---|---:|
| District | 0.043290 |
| crime_index | 0.885660 |
| Hour | 0.043673 |
| domestic_index | 0.027377 |

The most important feature was:

```text
crime_index
```

This means the type of crime was the strongest predictor of whether an arrest occurred.

In [ ]:
rf_model = trained_models["Random Forest"]

feature_names = [
    "District",
    "crime_index",
    "Hour",
    "domestic_index"
]

importances = rf_model.featureImportances.toArray()

print("Random Forest Feature Importances:")
for feature_name, importance in zip(feature_names, importances):
    print(f"{feature_name:<20} {importance:.6f}")

most_important_index = int(importances.argmax())
print("\nMost important feature:", feature_names[most_important_index])

# 12. Milestone 1 vs Milestone 2 Comparison

Milestone 1 used Hadoop MapReduce. Milestone 2 used Spark DataFrames.

| Analysis | Milestone 1 Approach | Milestone 2 Approach | Status |
|---|---|---|---|
| Crime type distribution | Hadoop MapReduce | Spark DataFrame `groupBy` | Completed |
| Location hotspots | Hadoop MapReduce | Spark DataFrame `groupBy` | Completed |
| Yearly crime trends | Hadoop MapReduce | Spark DataFrame `groupBy` | Completed |
| Arrest analysis | Hadoop MapReduce | Spark DataFrame `groupBy` | Completed |

Sample side-by-side results:

| Metric | Milestone 1 MapReduce Result | Milestone 2 Spark Result | Match Status |
|---|---:|---:|---|
| THEFT count | 2054 | 2054 | Matches |
| BATTERY count | 1728 | 1728 | Matches |
| CRIMINAL DAMAGE count | 1062 | 1062 | Matches |
| Arrest = true | 1283 | 1283 | Matches |
| Arrest = false | 8717 | 8717 | Matches |

# 13. Execution Evidence

The repository includes saved logs proving execution.

## Local Mode Evidence

File:

```text
output/milestone2/local_execution_output.txt
```

Important evidence:

```text
Spark Version: 3.5.4
Spark Master: local[*]
Application ID: local-1778420218938
local_execution_check.py completed successfully
```

## YARN ML Evidence

File:

```text
output/milestone2/m2_spark_ml_client_output.txt
```

Important evidence:

```text
Spark Version: 3.5.4
Spark Master: yarn
Application ID: application_1777830883738_0060
Input path: hdfs:///data/chicago_crimes_sample.csv
Total rows loaded: 10000
m2_spark_ml.py completed successfully
```

## Full Dataset YARN Evidence

File:

```text
output/milestone2/full_dataset_phaseA_yarn_output.txt
```

Important evidence:

```text
Spark Version: 3.5.4
Spark Master: yarn
Application ID: application_1777830883738_0061
Input path: hdfs:///data/chicago_crimes.csv
Total rows in full dataset: 793073
full_dataset_phaseA_check.py completed successfully
```

# 14. Challenges and Solutions

## Challenge 1: Python 3.12 Missing NumPy

The cluster default PySpark environment used Python 3.12, but Spark MLlib required NumPy.

### Solution

The Spark jobs were submitted using Python 3.10:

```bash
PYSPARK_PYTHON=/usr/bin/python3.10
PYSPARK_DRIVER_PYTHON=/usr/bin/python3.10
```

## Challenge 2: Cluster Memory Limits

Training multiple ML models and CrossValidator in one heavy job can exceed the YARN memory limits.

### Solution

The repository includes:

- a stable standalone Spark ML script,
- separate model scripts,
- sample dataset ML training,
- full dataset YARN analytics evidence.

## Challenge 3: Class Imbalance

The label distribution is imbalanced:

```text
No Arrest: 8717
Arrest:    1283
```

### Solution

Multiple metrics are reported:

- Accuracy
- F1 Score
- Precision
- Recall
- AUC
- Confusion Matrix

# 15. Conclusion

Milestone 2 successfully demonstrates Spark analytics and Spark MLlib modeling on the Chicago Crimes dataset.

The final implementation includes:

- Spark DataFrame analytics,
- corrected ML feature vector,
- three ML models,
- model evaluation metrics,
- confusion matrices,
- feature importance,
- local execution evidence,
- YARN execution evidence,
- full dataset YARN evidence.

The complete workflow is:

```text
HDFS → Spark DataFrames → Feature Engineering → Spark MLlib Models → Evaluation → Interpretation
```

In [ ]:
# Optional cleanup when finished.
train_data.unpersist()
test_data.unpersist()
spark.stop()
print("Notebook completed successfully.")